# Loss Given Default (LGD): Empirical Validation

**FYP: Explainable AI for Loan Default Prediction** — supporting analysis for `webapp/backend/prepare_artifacts.py`.

Most credit-risk projects simply cite an industry-benchmark range for LGD (e.g. "unsecured consumer
credit typically runs 65-85%"), since a single model's own training data rarely retains enough
resolved, recovered defaults to measure it directly — the working dataset (`loan_2014_20.csv`) is no
exception, and doesn't carry the payment/recovery fields (`total_rec_prncp`, `recoveries`,
`collection_recovery_fee`) needed to compute realized loss rates.

Rather than stop at a cited range, this notebook validates the assumption empirically: it
cross-references a broader LendingClub release — `accepted_2007_to_2018Q4.csv` — that *does* track
recovery history for the same unsecured personal loan population, and computes LGD directly from
269,320 loans that actually defaulted and went through collections. The result (**LGD ≈ 63%**) is
what `prepare_artifacts.py` and `main.py` use throughout the dashboard, replacing the earlier 70%
industry-benchmark starting point with a figure grounded in observed outcomes.

**Scope note**: this file's loan vintages (2007–2018Q4) don't exactly match the main project's
population (2014–2020) — there's no single LendingClub release that has both the recovery fields
*and* the project's exact date range. The overlap (2014–2018Q4) is substantial, and the by-vintage
breakdown below is included specifically to check whether that mismatch — or a more consequential
data artifact, right-censoring — biases the result.

## 1. Load the recovery-fields data

Only pull the columns needed — the full file is 1.6M rows, ~1.7GB on disk.

In [1]:
import pandas as pd
import numpy as np

COLS = ['loan_status', 'funded_amnt', 'total_rec_prncp', 'recoveries',
        'collection_recovery_fee', 'issue_d']

df = pd.read_csv('accepted_2007_to_2018Q4.csv', usecols=COLS, low_memory=False)
print(f"Total rows: {len(df):,}")
df.head()

Total rows: 2,260,701


,funded_amnt,issue_d,loan_status,total_rec_prncp,recoveries,collection_recovery_fee
0,3600.0,Dec-2015,Fully Paid,3600.00,0.0,0.0
1,24700.0,Dec-2015,Fully Paid,24700.00,0.0,0.0
2,20000.0,Dec-2015,Fully Paid,20000.00,0.0,0.0
3,35000.0,Dec-2015,Current,19102.35,0.0,0.0
4,10400.0,Dec-2015,Fully Paid,10400.00,0.0,0.0


## 2. Loan status breakdown

Only loans with a **resolved terminal default status** (`Charged Off`) have a final, measurable recovery — a loan that's merely `Late` or `In Grace Period` hasn't finished its collections process yet.

In [2]:
df['loan_status'].value_counts(dropna=False)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

## 3. Compute per-loan LGD

Standard definition: `LGD = loss / exposure`. This project computes expected loss as
`calibrated_pd × loan_amnt × LGD` — i.e. exposure is the **original loan amount at origination**,
not the outstanding balance at the moment of default (which isn't knowable in advance, and is what
the model is scoring *before* any repayment has happened). So LGD here is measured on that same basis:

```
LGD = (funded_amnt − total_rec_prncp − net_recovery) / funded_amnt
```

- `total_rec_prncp` — principal the borrower repaid over the life of the loan, before charge-off.
- `net_recovery = recoveries − collection_recovery_fee` — post-charge-off collections proceeds,
  net of the fee paid to the collection agency (recoveries is the gross amount collected; the fee
  is the lender's cost of collecting it, so it should reduce what's actually recovered).

Clipped to `[0, 1]` — a handful of loans have `total_rec_prncp + net_recovery` slightly exceed
`funded_amnt` (partial overpayment / rounding), which isn't a real >100% loss.

In [3]:
DEFAULTED_STATUSES = ['Charged Off', 'Does not meet the credit policy. Status:Charged Off']

d = df[df['loan_status'].isin(DEFAULTED_STATUSES)].copy()
print(f"Resolved charged-off loans: {len(d):,}")

d['net_recovery'] = d['recoveries'] - d['collection_recovery_fee']
d['net_loss'] = d['funded_amnt'] - d['total_rec_prncp'] - d['net_recovery']
d['lgd_raw'] = d['net_loss'] / d['funded_amnt']

print()
print("Before clipping:")
print(d['lgd_raw'].describe())
print(f"share < 0 (overpayment/rounding): {(d['lgd_raw'] < 0).mean():.4%}")
print(f"share > 1: {(d['lgd_raw'] > 1).mean():.4%}")

d['lgd'] = d['lgd_raw'].clip(0, 1)

Resolved charged-off loans: 269,320

Before clipping:
count    269320.000000
mean          0.634398
std           0.215057
min          -1.170300
25%           0.497344
50%           0.676457
75%           0.801428
max           1.000000
Name: lgd_raw, dtype: float64
share < 0 (overpayment/rounding): 0.0323%
share > 1: 0.0000%


## 4. Headline result

In [4]:
unweighted_mean = d['lgd'].mean()
weighted_mean = (d['lgd'] * d['funded_amnt']).sum() / d['funded_amnt'].sum()
median_lgd = d['lgd'].median()

print(f"Unweighted mean LGD:            {unweighted_mean:.1%}")
print(f"Funded-amount-weighted mean LGD: {weighted_mean:.1%}   <- portfolio-level: total $ lost / total $ funded")
print(f"Median LGD:                      {median_lgd:.1%}")

Unweighted mean LGD:            63.4%
Funded-amount-weighted mean LGD: 65.3%   <- portfolio-level: total $ lost / total $ funded
Median LGD:                      67.6%


## 5. Checking for right-censoring bias by vintage

Post-charge-off recoveries can trickle in for months or years after the charge-off date. Loans that
charged off close to this file's snapshot (late 2018) haven't had that process finish, so their
*measured* LGD will look artificially high — not because recovery was actually worse, just because
it wasn't done yet when this snapshot was taken. Breaking out by `issue_d` year checks for this.

In [5]:
d['issue_year'] = pd.to_datetime(d['issue_d'], format='%b-%Y').dt.year

by_year = d.groupby('issue_year').apply(
    lambda x: pd.Series({
        'n': len(x),
        'mean_lgd': x['lgd'].mean(),
        'weighted_lgd': (x['lgd'] * x['funded_amnt']).sum() / x['funded_amnt'].sum(),
    }),
    include_groups=False,
)
by_year

,n,mean_lgd,weighted_lgd
issue_year,,,
2007,158.0,0.570097,0.586175
2008,496.0,0.596405,0.619001
2009,723.0,0.598778,0.611940
2010,1757.0,0.589577,0.594916
2011,3297.0,0.600021,0.613190
2012,8644.0,0.566778,0.579612
2013,21024.0,0.554185,0.564135
2014,41161.0,0.565408,0.579856
2015,75803.0,0.597590,0.618787


LGD climbs sharply for 2017–2018 vintages (mid-70s to 90%) versus a stable ~55–62% for the mature
2012–2015 vintages. That's the right-censoring effect described above, not a real deterioration in
recovery rates — those recent charge-offs simply haven't finished collections yet at the time this
data was pulled. The mature vintages are the more reliable read on the long-run LGD.

## 6. Restrict to the project's overlap window (2014–2018Q4)

For comparison, and because it's the closest match to `loan_2014_20.csv`'s population that this file
can provide (it stops at 2018Q4, so 2019–2020 vintages aren't available here at all).

In [6]:
overlap = d[d['issue_year'] >= 2014]
overlap_weighted = (overlap['lgd'] * overlap['funded_amnt']).sum() / overlap['funded_amnt'].sum()

print(f"Charged-off loans issued 2014-2018Q4: {len(overlap):,}")
print(f"Unweighted mean LGD: {overlap['lgd'].mean():.1%}")
print(f"Weighted mean LGD:   {overlap_weighted:.1%}")
print(f"Median LGD:          {overlap['lgd'].median():.1%}")

Charged-off loans issued 2014-2018Q4: 233,221
Unweighted mean LGD: 64.5%
Weighted mean LGD:   66.5%
Median LGD:          68.7%


## 7. Conclusion

| Cut | Weighted mean LGD |
|---|---|
| All vintages (2007–2018Q4) | 65.3% |
| 2014–2018Q4 (overlaps this project's data) | 66.5% |
| Mature vintages only (2012–2015, unaffected by censoring) | 56–62% |

The pooled, all-vintage figure is pulled upward by recently-charged-off loans whose recoveries
haven't finished processing. Discounting that bias, the real LGD for LendingClub unsecured personal
loans settles in the **high-50s to mid-60s %** — meaningfully below the 70% industry-benchmark
starting point this project used prior to this validation, but in the same ballpark (70% wasn't
wrong, just the conservative end of a range, whereas this is a value derived from actual defaulted
loans and their recoveries).

**Value adopted**: `LGD = 0.63` — the midpoint of the credible range, and close to both the
2014-2018Q4-restricted weighted mean (66.5%) and the mature-vintage weighted range (56–62%). This is
now set as `LGD` in `webapp/backend/prepare_artifacts.py` and `LGD_DEFAULT` in `webapp/backend/main.py`,
replacing the old 70% assumption. Changing it shifted the cost-minimizing decision threshold from
0.4388 to 0.4838 (a lower LGD makes missed defaults cheaper relative to false-positive declines, which
pushes the optimal threshold higher / more conservative about declining).